<a href="https://colab.research.google.com/github/chaunijs/onlineshoppingprice/blob/main/Tops_json_gem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Run Prompt in Gemini Pro
You are an expert data extraction assistant. I will provide you with raw text or HTML scraped from the Tops Supermarket e-commerce website (Fabric Softener category).



Your task is to parse the content and extract the product details into a clean, structured JSON format.

Please extract the following 4 fields for every product you find:

1. "name" (Product name): The full name of the product including volume/size if present (e.g., "Downy Concentrated Fabric Softener Sunrise Fresh 2ltr. Refill").

2. "promo_price" (Promo price): The current discounted price or checkout price of the item. Only include the numeric value (e.g., 189.00). Note: The raw text often duplicates numbers like "฿189.00189" — clean this up to just "189.00".

3. "original_price" (Normal price): The original/strikethrough price before discount. Only include the numeric value (e.g., 210.00). Clean up anomalies like "฿210.00210" to just "210.00". If there is no normal price listed, output null.

4. "condition" (Condition/Promotion): Any promotional tag applied to the item. Look specifically for phrases like "Buy 2 Pay 1", "Buy 2 Save More", "RedHot", "Sale", or "Save ฿X.XX". If there is no promotional condition, output null.



### Rules for Extraction:

- Do not invent or hallucinate data. If a field is missing, use null.

- Clean the pricing data so it is formatted as a standard float/decimal (e.g., 189.00, not ฿189.00 or 189.00189).

- Exclude generic website navigation text, category names, or unrelated numbers.



### Desired JSON Output Format:

[

  {

    "name": "Fineline Concentrated Fabric Softener Premium Organic Green 1000ml.",

    "normal_price": 195.00,

    "selling_price": 194.00,

    "condition": "Buy 2 Pay 1"

  },

  ...

]



Here is the raw data to process:

watchlist = [

    "https://www.tops.co.th/en/fineline-liquid-detergent-sunny-gold-550ml-8851989033365",

    "https://www.tops.co.th/en/fineline-liquid-detergent-sunny-gold-1250ml-8851989034737",

    "https://www.tops.co.th/en/hygiene-expert-wash-concentrated-liquid-detergent-milky-touch-scent-600ml-8850092254155",

    "https://www.tops.co.th/en/hygiene-expert-wash-concentrated-liquid-detergent-milky-touch-scent-1400ml-8850092254216",

    "https://www.tops.co.th/en/pao-win-wash-liquid-concentrated-detergent-620ml-refill-8850002024823",

    "https://www.tops.co.th/en/pao-win-wash-liquid-concentrated-detergent-1300ml-refill-8850002031739",

    "https://www.tops.co.th/en/hygiene-expert-care-concentrate-fabric-softener-milky-touch-white-480ml-8850092280604",

    "https://www.tops.co.th/en/hygiene-expert-care-concentrate-fabric-softener-milky-touch-480ml-pack-3-8850092280819",

    "https://www.tops.co.th/en/hygiene-expert-care-concentrate-fabric-softener-milky-touch-white-1000ml-8850092280901",

    "https://www.tops.co.th/en/hygiene-expert-care-concentrate-fabric-softener-milky-touch-white-1000ml-pack-2-8850092280925",

    "https://www.tops.co.th/en/lipon-f-x-tra-hygienic-dish-wash-500ml-pack-3-8850002042643",

    "https://www.tops.co.th/en/lipon-f-dish-wash-3-2ltr-8850002010772"

]



In [1]:
! uv pip install xlsxwriter polars

Using Python 3.12.13 environment at: /usr
Resolved 3 packages in 275ms
Prepared 1 package in 49ms
Installed 1 package in 3ms
 + xlsxwriter==3.2.9


In [28]:
# Paste HERE
copied_text = """
[
  {
    "name": "Fineline Liquid Detergent Sunny Gold 550ml",
    "original_price": null,
    "promotion_price": 89.00,
    "condition": "Buy 2 Pay 1"
  },
  {
    "name": "Fineline Liquid Detergent Sunny Gold 1250ml",
    "original_price": null,
    "promotion_price": 179.00,
    "condition": "Buy 2 Pay 1"
  },
  {
    "name": "Hygiene Expert Wash Concentrated Liquid Detergent Milky Touch Scent 600ml",
    "original_price": 65.00,
    "promotion_price": 55.00,
    "condition": "Sale"
  },
  {
    "name": "Hygiene Expert Wash Concentrated Liquid Detergent Milky Touch Scent 1400ml",
    "original_price": 139.00,
    "promotion_price": 109.00,
    "condition": "RedHot"
  },
  {
    "name": "Pao Win Wash Liquid Concentrated Detergent 620ml Refill",
    "original_price": null,
    "promotion_price": 99.00,
    "condition": null
  },
  {
    "name": "Pao Win Wash Liquid Concentrated Detergent 1300ml Refill",
    "original_price": null,
    "promotion_price": 185.00,
    "condition": "Buy 2 Pay 1"
  },
  {
    "name": "Hygiene Expert Care Concentrate Fabric Softener Milky Touch White 480ml",
    "original_price": 60.00,
    "promotion_price": 49.00,
    "condition": "Sale"
  },
  {
    "name": "Hygiene Expert Care Concentrate Fabric Softener Milky Touch 480ml Pack 3",
    "original_price": null,
    "promotion_price": 120.00,
    "condition": null
  },
  {
    "name": "Hygiene Expert Care Concentrate Fabric Softener Milky Touch White 1000ml",
    "original_price": 129.00,
    "promotion_price": 109.00,
    "condition": "Sale"
  },
  {
    "name": "Hygiene Expert Care Concentrate Fabric Softener Milky Touch White 1000ml Pack 2",
    "original_price": null,
    "promotion_price": 219.00,
    "condition": null
  },
  {
    "name": "Lipon F X Tra Hygienic Dish Wash 500ml Pack 3",
    "original_price": 87.00,
    "promotion_price": 69.00,
    "condition": "Sale"
  },
  {
    "name": "Lipon F Dish Wash 3.2ltr",
    "original_price": null,
    "promotion_price": 165.00,
    "condition": null
  }
]
"""

In [29]:
import polars as pl
import io

schema = {
    "name": pl.Utf8,
    "original_price": pl.Float64, # Changed from original_price
    "promotion_price": pl.Float64, # Changed from promo_price
    "condition": pl.Utf8
}
df = pl.read_json(io.BytesIO(copied_text.encode('utf-8')), schema=schema)

In [30]:
df

name,original_price,promotion_price,condition
str,f64,f64,str
"""Fineline Liquid Detergent Sunn…",null,89.0,"""Buy 2 Pay 1"""
"""Fineline Liquid Detergent Sunn…",null,179.0,"""Buy 2 Pay 1"""
"""Hygiene Expert Wash Concentrat…",65.0,55.0,"""Sale"""
"""Hygiene Expert Wash Concentrat…",139.0,109.0,"""RedHot"""
"""Pao Win Wash Liquid Concentrat…",null,99.0,null
…,…,…,…
"""Hygiene Expert Care Concentrat…",null,120.0,null
"""Hygiene Expert Care Concentrat…",129.0,109.0,"""Sale"""
"""Hygiene Expert Care Concentrat…",null,219.0,null


In [31]:
# @title revaluate price
def re_evaluate_price(df: pl.DataFrame) -> pl.DataFrame:
    """
    Standardizes pricing logic:
    1. Swaps original and promotion if original < promotion.
    2. If original_price is missing, fills it with promotion_price.
    3. If promotion_price matches original_price, sets promotion to Null.
    """
    return (
        df.with_columns(
            # Create temporary columns to determine the true Max and Min
            # This automatically handles the "swap" and "missing original" logic
            pl.max_horizontal("original_price", "promotion_price").alias("original_price"),
            pl.min_horizontal("original_price", "promotion_price").alias("temp_promo")
        )
        .with_columns(
            # Finalize promotion_price: if it equals original_price, it's not a promotion
            pl.when(pl.col("temp_promo") == pl.col("original_price"))
            .then(None)
            .otherwise(pl.col("temp_promo"))
            .alias("promotion_price")
        )
        .drop("temp_promo") # Clean up helper column
    )

In [22]:
import datetime
from datetime import date
today_date = datetime.datetime.now().strftime("%Y-%m-%d")
print("Today is",today_date)

Today is 2026-06-16


In [26]:
# @title udf Transform
def parse_product_names(df: pl.DataFrame, shop_name: str) -> pl.DataFrame:
    """
    Pass any supermarket dataframe through this function to standardize the columns.
    """
    # 1. Setup the dynamic date
    today_date = date.today().strftime("%Y-%m-%d")

    # 2. Define the patterns
    # FIXED: Added LTR and LITERS? to the unit options
    quant_unit_pattern = r"(?i)([\d.]+)\s*(ML|G|KG|L|LTR|LITERS?|GRAMS?)"

    # Keeping the fix from the previous pack pattern
    pack_pattern = r"(?i)(PACK\s*\d*\s*FREE\s*\d+|PACK\s*\d*\s*\+\s*\d+|PACK\s*\d+|TWINPACK|\bX\s*\d+\s*\+\s*\d+|\bX\s*\d+\b|P?\d+\s*\+\s*\d+||\d+\s*FREE\s*\d+|\bPACK\b)"

    # 3. Apply the Polars transformation
    return df.with_columns(
        pl.lit(today_date).alias("Date"),

        # Extract Brand
        pl.col("name").str.split(" ").list.first().alias("Brand"),

        # Extract Quantity
        # FIXED: Changed pl.Int16 to pl.Float64 so 3.2 is captured properly
        pl.col("name").str.extract(quant_unit_pattern, 1).cast(pl.Float64, strict=False).alias("Volume"),

        # Extract Unit
        pl.col("name").str.extract(quant_unit_pattern, 2).str.to_uppercase().alias("Unit"),

        # Extract Pack size
        pl.col("name").str.extract(pack_pattern, 1).str.to_uppercase().alias("Pack"),

        # Add the dynamic Shop identifier
        pl.lit(shop_name).alias("Retailer")
    )

In [32]:
# Usage:
df_prep_tops_watchlist = re_evaluate_price(df)
df_trans_tops_watchlist = parse_product_names(df_prep_tops_watchlist, "Tops")
# saving Excel file
df_trans_tops_watchlist.unique().write_excel(f"tops_watchlists_{today_date}.xlsx")

In [33]:
df_trans_tops_watchlist.to_pandas()

,name,original_price,promotion_price,condition,Date,Brand,Volume,Unit,Pack,Retailer
0,Fineline Liquid Detergent Sunny Gold 550ml,89.0,NaN,Buy 2 Pay 1,2026-06-16,Fineline,550.0,ML,,Tops
1,Fineline Liquid Detergent Sunny Gold 1250ml,179.0,NaN,Buy 2 Pay 1,2026-06-16,Fineline,1250.0,ML,,Tops
2,Hygiene Expert Wash Concentrated Liquid Deterg...,65.0,55.0,Sale,2026-06-16,Hygiene,600.0,ML,,Tops
3,Hygiene Expert Wash Concentrated Liquid Deterg...,139.0,109.0,RedHot,2026-06-16,Hygiene,1400.0,ML,,Tops
4,Pao Win Wash Liquid Concentrated Detergent 620...,99.0,NaN,None,2026-06-16,Pao,620.0,ML,,Tops
5,Pao Win Wash Liquid Concentrated Detergent 130...,185.0,NaN,Buy 2 Pay 1,2026-06-16,Pao,1300.0,ML,,Tops
6,Hygiene Expert Care Concentrate Fabric Softene...,60.0,49.0,Sale,2026-06-16,Hygiene,480.0,ML,,Tops
7,Hygiene Expert Care Concentrate Fabric Softene...,120.0,NaN,None,2026-06-16,Hygiene,480.0,ML,,Tops
8,Hygiene Expert Care Concentrate Fabric Softene...,129.0,109.0,Sale,2026-06-16,Hygiene,1000.0,ML,,Tops
9,Hygiene Expert Care Concentrate Fabric Softene...,219.0,NaN,None,2026-06-16,Hygiene,1000.0,ML,,Tops
